# Revision round 4: analysis of the rerun experiments

This notebook summarizes the experiments requested by Reviewer 2 (Complexity, manuscript 5411286).
The raw per-fold outputs were produced by `scripts/run_round4_kaggle.py` and `scripts/run_round4_ulb.py`
(console logs in `logs/`). Here they are aggregated into the values of Tables 7 to 13 and the statistical tests.

* Kaggle CreditCard-Fraud-Detection data: `credit_card` key excluded, `State` encoded from the state names,
  all seven columns standardized, one feature matrix for all four algorithms.
* ULB European benchmark (284,807 transactions, 492 frauds): labels used only for evaluation.

In [1]:
import json, platform, sys
from pathlib import Path
import numpy as np, pandas as pd, sklearn, scipy
from scipy import stats
sys.path.insert(0, str(Path('..') / 'src'))
from ccf_clustering import evaluation as ev
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
R = Path('..') / 'results' / 'round4'
print(platform.python_version(), 'numpy', np.__version__, 'pandas', pd.__version__,
      'scikit-learn', sklearn.__version__, 'scipy', scipy.__version__)
S = {}   # all numbers used in the manuscript
MODELS = ['KMC', 'AHC', 'GMMC', 'FCMC']

3.13.18 numpy 2.5.3 pandas 3.0.5 scikit-learn 1.9.1 scipy 1.18.1


## Table 7: four algorithms, K = 2, same standardized feature matrix (10-fold)

In [2]:
main = pd.read_csv(R / 'main_K2_per_fold.csv')
agg = main.groupby('model')[['SS', 'CHI', 'DBI', 'ARI', 'min_cluster_share']].agg(['mean', 'std']).loc[MODELS]
t7 = pd.DataFrame({m: {k: agg.loc[m, (k, 'mean')] for k in ['SS', 'CHI', 'DBI', 'ARI', 'min_cluster_share']} for m in MODELS}).T
t7['mean_rank'] = ev.mean_rank(t7)
for m in MODELS:
    S[f't7_{m}'] = {k: float(agg.loc[m, (k, s)]) for k in ['SS', 'CHI', 'DBI', 'ARI', 'min_cluster_share'] for s in ['mean']}
    S[f't7_{m}'].update({k + '_sd': float(agg.loc[m, (k, 'std')]) for k in ['SS', 'CHI', 'DBI', 'ARI']})
    S[f't7_{m}']['mean_rank'] = float(t7.loc[m, 'mean_rank'])
    S[f't7_{m}']['SS_range'] = [float(main[main.model == m].SS.min()), float(main[main.model == m].SS.max())]
    S[f't7_{m}']['ARI_range'] = [float(main[main.model == m].ARI.min()), float(main[main.model == m].ARI.max())]
t7.round(4)

,SS,CHI,DBI,ARI,min_cluster_share,mean_rank
KMC,0.3968,70107.5147,1.3423,0.9999,0.1796,1.6667
AHC,0.6462,29826.4250,1.0553,0.6152,0.0161,2.0000
GMMC,0.2085,30708.3483,2.4453,0.9172,0.4474,3.6667
FCMC,0.2795,56006.1709,1.7333,0.9986,0.4599,2.6667


### Friedman test and Holm-corrected Wilcoxon signed-rank tests across the ten folds

In [3]:
pf = {m: main[main.model == m].sort_values('fold').reset_index(drop=True) for m in MODELS}
S['friedman_kaggle'] = {}
for metric in ['SS', 'CHI', 'DBI']:
    chi2, p, pairs = ev.friedman_wilcoxon(pf, metric, MODELS)
    S['friedman_kaggle'][metric] = {'chi2': float(chi2), 'p': float(p),
                                    'max_p_holm': float(pairs.p_holm.max())}
    print(metric, 'Friedman chi2 = %.2f, p = %.2e' % (chi2, p)); print(pairs.round(4))
# KMC vs AHC on each index, number of folds in which KMC is better
S['kmc_better_than_ahc_CHI_folds'] = int((pf['KMC'].CHI > pf['AHC'].CHI).sum())

SS Friedman chi2 = 28.92, p = 2.33e-06
  model_a model_b  p_wilcoxon  p_holm
0     KMC     AHC      0.0020  0.0117
1     KMC    GMMC      0.0020  0.0117
2     KMC    FCMC      0.0020  0.0117
3     AHC    GMMC      0.0020  0.0117
4     AHC    FCMC      0.0020  0.0117
5    GMMC    FCMC      0.0098  0.0117
CHI Friedman chi2 = 27.00, p = 5.89e-06
  model_a model_b  p_wilcoxon  p_holm
0     KMC     AHC      0.0020  0.0117
1     KMC    GMMC      0.0020  0.0117
2     KMC    FCMC      0.0020  0.0117
3     AHC    GMMC      0.6953  0.6953
4     AHC    FCMC      0.0020  0.0117
5    GMMC    FCMC      0.0020  0.0117
DBI Friedman chi2 = 28.08, p = 3.49e-06
  model_a model_b  p_wilcoxon  p_holm
0     KMC     AHC      0.0098  0.0117
1     KMC    GMMC      0.0020  0.0117
2     KMC    FCMC      0.0020  0.0117
3     AHC    GMMC      0.0020  0.0117
4     AHC    FCMC      0.0020  0.0117
5    GMMC    FCMC      0.0020  0.0117


### Table 9: KMC per fold

In [4]:
t9 = pf['KMC'][['fold', 'SS', 'CHI', 'DBI', 'ARI']]
S['t9'] = t9.to_dict('list')
t9.round(4)

,fold,SS,CHI,DBI,ARI
0,1,0.3989,69688.0757,1.3433,0.9999
1,2,0.3962,70140.6217,1.3428,1.0000
2,3,0.3981,69924.2652,1.3443,0.9998
3,4,0.3992,70296.8497,1.3411,1.0000
4,5,0.3963,69965.3308,1.3425,1.0000
5,6,0.3950,70253.0080,1.3417,1.0000
6,7,0.3974,69950.3330,1.3431,1.0000
7,8,0.3992,70047.0656,1.3431,0.9999
8,9,0.3957,70449.5255,1.3401,0.9999
9,10,0.3919,70360.0717,1.3408,0.9999


### Accuracy of the subsampled Silhouette Score (training fold 1, KMC)

In [5]:
chk = json.load(open(R / 'ss_exact_check_fold1.json')); S['ss_check'] = chk['KMC']; chk

{'KMC': {'exact': 0.39758608956204156,
  'approx_seed42': 0.3989141177359986,
  'approx_mean_10_subsamples': 0.3973640995372648,
  'approx_sd_10_subsamples': 0.0018710806421067747,
  'sec_exact': 754.58}}

## Table 8: K = 2 to 6 for all four algorithms

In [6]:
ks = pd.read_csv(R / 'k_sweep_per_fold.csv')
t8 = ks.groupby(['model', 'K'])[['SS', 'CHI', 'DBI', 'ARI', 'min_cluster_share']].mean()
t8['rank_within_model'] = np.nan
for m in MODELS:
    sub = t8.loc[m]
    t8.loc[m, 'rank_within_model'] = ev.mean_rank(sub).values
t8 = t8.reindex(MODELS, level=0)
S['t8'] = {f'{m}_K{k}': {c: float(t8.loc[(m, k), c]) for c in t8.columns} for m, k in t8.index}
ib = pd.read_csv(R / 'k_sweep_inertia_bic.csv'); S['inertia_bic'] = ib.to_dict('list')
display(t8.round(4)); ib

SS         CHI     DBI     ARI  min_cluster_share  rank_within_model
model K                                                                          
KMC   2  0.3968  70107.5147  1.3423  0.9999             0.1796             2.6667
      3  0.3065  65176.1650  1.4179  0.9998             0.1744             5.0000
      4  0.3280  68963.1993  1.1481  0.9997             0.0186             3.6667
      5  0.3389  77411.5299  1.0535  0.9998             0.0097             1.6667
      6  0.3364  75984.4519  1.0535  0.9612             0.0096             2.0000
AHC   2  0.6462  29826.4250  1.0553  0.6152             0.0161             3.6667
      3  0.5651  38843.6219  1.0450  0.7158             0.0131             3.3333
      4  0.4573  53729.7713  1.0221  0.7645             0.0112             2.3333
      5  0.4227  52932.6694  0.9729  0.8893             0.0044             2.3333
      6  0.3525  51948.7621  1.0003  0.7534             0.0033             3.3333
GMMC  2  0.2085  30708.3483  2.4453  0.9172             0.4474             5.0000
      3  0.2756  55375.5538  1.5684  0.9125             0.1772             2.3333
      4  0.2847  53306.4293  1.6330  0.9707             0.0345             2.6667
      5  0.3030  60426.4614  1.4125  0.9236             0.0180             1.0000
      6  0.2720  49008.9358  1.8707  0.8530             0.0179             4.0000
FCMC  2  0.2795  56006.1709  1.7333  0.9986             0.4599             3.0000
      3  0.3055  65039.3334  1.4132  0.9997             0.1752             1.0000
      4  0.2892  55506.7465  1.4563  0.9967             0.1729             2.3333
      5  0.2357  45623.8409  1.6293  0.9981             0.1609             3.6667
      6  0.2225  39213.8369  1.9798  0.9984             0.1075             5.0000

,K,KMC_inertia,GMMC_BIC
0,1,2.062116e+06,5.614816e+06
1,2,1.630870e+06,3.979120e+05
2,3,1.382435e+06,-3.621877e+05
3,4,1.158271e+06,-1.732184e+06
4,5,9.511993e+05,-2.824990e+06
5,6,8.471444e+05,-2.128127e+06


## Table 10: hyperparameter sensitivity at K = 2, seed stability, and feature-set sensitivity

In [7]:
hy = pd.read_csv(R / 'hyper_K2_per_fold.csv')
t10 = hy.groupby(['model', 'config'], sort=False)[['SS', 'CHI', 'DBI', 'ARI', 'min_cluster_share']].mean()
S['t10'] = {c: {k: float(v) for k, v in row.items()} for (m, c), row in t10.iterrows()}
seeds = pd.read_csv(R / 'seed_stability.csv'); S['seeds'] = seeds.set_index('model').to_dict('index')
fs = pd.read_csv(R / 'features_continuous5_per_fold.csv')
t_fs = fs.groupby('model')[['SS', 'CHI', 'DBI', 'ARI', 'min_cluster_share']].mean().loc[MODELS]
t_fs['mean_rank'] = ev.mean_rank(t_fs)
S['features5'] = t_fs.to_dict('index')
display(t10.round(4)); display(seeds.round(4)); t_fs.round(4)

SS         CHI     DBI     ARI  min_cluster_share
model config                                                                                     
KMC   KMC | init=random, n_init=10          0.3968  70107.5179  1.3423  0.9999             0.1796
      KMC | init=k-means++, n_init=1        0.2968  57979.3037  1.7140  0.9277             0.3853
AHC   AHC | linkage=complete, 50 centroids  0.6748  27498.5058  1.0847  0.7027             0.0138
      AHC | linkage=average, 50 centroids   0.7649  21154.3079  0.5060  0.8842             0.0033
      AHC | linkage=ward, 200 centroids     0.6343  32472.5682  0.9907  0.5911             0.0284
GMMC  GMMC | covariance=tied                0.2828  50360.6012  1.8040  0.9117             0.3841
      GMMC | covariance=diag                0.2030  42181.4836  1.7083  0.8950             0.3594
      GMMC | covariance=spherical           0.1980  41385.5206  1.7369  0.9977             0.4068
FCMC  FCMC | m=1.5                          0.2867  56544.3973  1.7518  0.9971             0.4298
      FCMC | m=2.5                          0.2759  55657.1662  1.7270  0.9990             0.4704
      FCMC | m=3.0                          0.2738  55477.8753  1.7232  0.9989             0.4761

,model,ARI_mean,ARI_min
0,KMC,1.0000,1.0000
1,AHC,0.2382,-0.0193
2,GMMC,0.6033,0.2972
3,FCMC,1.0000,1.0000


,SS,CHI,DBI,ARI,min_cluster_share,mean_rank
model,,,,,,
KMC,0.4563,76417.3850,1.1494,0.9999,0.1827,2.0
AHC,0.7281,41881.8341,1.0232,0.7285,0.0164,2.0
GMMC,0.5310,46241.1499,1.9281,0.9053,0.1484,3.0
FCMC,0.4472,71843.7265,1.3999,0.9984,0.2332,3.0


## Table 11: fraud-risk candidates on the Kaggle data (no labels)

In [8]:
sc = pd.read_csv(R / 'scoring_summary.csv'); S['scoring'] = sc.to_dict('records')
ov = pd.read_csv(R / 'scoring_overlap.csv'); S['overlap'] = ov.to_dict('records')
rho = pd.read_csv(R / 'scoring_spearman.csv', index_col=0); S['score_spearman'] = rho.to_dict()
display(sc.round(3)); display(ov.round(3)); rho.round(3)

,model,whisker,n_flagged,pct_flagged,cards_with_flags,median_amount_flagged,median_amount_other,median_km_flagged,median_km_other,median_amt_to_cardmed_flagged,median_amt_to_cardmed_other,share_km_gt_100_flagged,share_km_gt_100_other
0,KMC,1.5,16257,5.519,932.0,145.480,56.76,7.256,6.071,1.439,0.988,0.256,0.000
1,KMC,3.0,9638,3.272,927.0,820.940,57.21,7.932,6.082,7.427,0.989,0.386,0.002
2,AHC,1.5,17236,5.851,901.0,102.215,57.26,6.261,6.114,1.346,0.989,0.015,0.014
3,AHC,3.0,6305,2.140,896.0,883.320,57.41,6.153,6.122,12.417,0.989,0.010,0.014
4,GMMC,1.5,18484,6.275,936.0,128.735,56.65,7.102,6.068,1.478,0.987,0.220,0.000
5,GMMC,3.0,10607,3.601,931.0,805.950,57.23,7.596,6.083,6.204,0.989,0.346,0.002
6,FCMC,1.5,19791,6.718,932.0,144.750,56.20,6.871,6.076,1.311,0.988,0.208,0.000
7,FCMC,3.0,9685,3.288,920.0,187.760,57.32,7.998,6.082,1.705,0.992,0.396,0.001
8,FCMC (1 - max u),1.5,0,0.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,IsolationForest,1.5,2939,0.998,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,a,b,jaccard
0,KMC,AHC,0.264
1,KMC,GMMC,0.692
2,KMC,FCMC,0.493
3,KMC,IF,0.164
4,AHC,GMMC,0.350
5,AHC,FCMC,0.298
6,AHC,IF,0.096
7,GMMC,FCMC,0.432
8,GMMC,IF,0.157
9,FCMC,IF,0.146


,score_KMC,score_AHC,score_GMMC,score_FCMC,IF
score_KMC,1.000,0.889,0.732,0.862,0.794
score_AHC,0.889,1.000,0.700,0.890,0.785
score_GMMC,0.732,0.700,1.000,0.861,0.955
score_FCMC,0.862,0.890,0.861,1.000,0.927
IF,0.794,0.785,0.955,0.927,1.000


## Table 12: detection performance on the labeled ULB benchmark (stratified 10-fold, K = 2)

In [9]:
u = pd.read_csv(R / 'ulb_per_fold_K2.csv')
DET = ['KMC', 'AHC', 'GMMC', 'FCMC', 'IF']
cols = ['precision', 'recall', 'f1', 'roc_auc', 'pr_auc', 'flag_rate',
        'w3_precision', 'w3_recall', 'w3_f1', 'w3_flag_rate',
        'top1_precision', 'top1_recall', 'top1_f1', 'top1_flag_rate', 'top05_precision', 'top05_recall', 'top05_f1', 'top05_flag_rate']
t12m = u.groupby('model')[cols].mean().loc[DET]; t12s = u.groupby('model')[cols].std().loc[DET]
S['t12'] = {m: {**{c: float(t12m.loc[m, c]) for c in cols}, **{c + '_sd': float(t12s.loc[m, c]) for c in cols}} for m in DET}
S['ulb_prevalence'] = 492 / 284807
for m in MODELS:
    S['t12'][m]['SS'] = float(u[u.model == m].SS.mean())
    S['t12'][m]['ARI_labels'] = float(u[u.model == m].ARI_labels.mean())
S['fcmc_ambiguity_ulb'] = {'roc_auc': float(u[u.model == 'FCMC']['roc_auc_1-maxu'].mean()),
                           'pr_auc': float(u[u.model == 'FCMC']['pr_auc_1-maxu'].mean())}
display(t12m.round(4)); t12s.round(4)

,precision,recall,f1,roc_auc,pr_auc,flag_rate,w3_precision,w3_recall,w3_f1,w3_flag_rate,top1_precision,top1_recall,top1_f1,top1_flag_rate,top05_precision,top05_recall,top05_f1,top05_flag_rate
model,,,,,,,,,,,,,,,,,,
KMC,0.0322,0.8740,0.0621,0.9548,0.1745,0.0469,0.0601,0.8051,0.1118,0.0232,0.1057,0.6119,0.1802,0.01,0.1538,0.4450,0.2285,0.0050
AHC,0.0296,0.7871,0.0570,0.9533,0.1607,0.0462,0.0565,0.7202,0.1048,0.0222,0.1002,0.5756,0.1706,0.01,0.1301,0.3703,0.1925,0.0050
GMMC,0.0203,0.8394,0.0396,0.9380,0.1934,0.0722,0.0397,0.6932,0.0748,0.0311,0.1049,0.6057,0.1787,0.01,0.1616,0.4693,0.2402,0.0050
FCMC,0.0306,0.8739,0.0590,0.9542,0.1739,0.0495,0.0577,0.8131,0.1077,0.0244,0.1057,0.6119,0.1802,0.01,0.1541,0.4450,0.2288,0.0050
IF,0.0300,0.8456,0.0580,0.9504,0.1889,0.0488,0.0721,0.7136,0.1309,0.0172,0.1013,0.5855,0.1727,0.01,0.1517,0.4432,0.2256,0.0051


,precision,recall,f1,roc_auc,pr_auc,flag_rate,w3_precision,w3_recall,w3_f1,w3_flag_rate,top1_precision,top1_recall,top1_f1,top1_flag_rate,top05_precision,top05_recall,top05_f1,top05_flag_rate
model,,,,,,,,,,,,,,,,,,
KMC,0.0017,0.0230,0.0031,0.0108,0.0316,0.0019,0.0044,0.0451,0.0079,0.0009,0.0068,0.0408,0.0112,0.0005,0.0182,0.0573,0.0269,0.0003
AHC,0.0074,0.1878,0.0143,0.0129,0.0632,0.0018,0.0167,0.2011,0.0308,0.0010,0.0198,0.1058,0.0332,0.0005,0.0472,0.1261,0.0686,0.0003
GMMC,0.0018,0.0716,0.0034,0.0183,0.0874,0.0095,0.0096,0.1232,0.0172,0.0068,0.0162,0.0996,0.0276,0.0005,0.0429,0.1381,0.0646,0.0004
FCMC,0.0020,0.0212,0.0038,0.0120,0.0314,0.0022,0.0041,0.0373,0.0075,0.0009,0.0067,0.0408,0.0110,0.0005,0.0184,0.0573,0.0271,0.0003
IF,0.0019,0.0302,0.0036,0.0125,0.0579,0.0025,0.0089,0.0621,0.0156,0.0016,0.0115,0.0604,0.0191,0.0006,0.0245,0.0738,0.0354,0.0005


In [10]:
upf = {m: u[u.model == m].sort_values('fold').reset_index(drop=True) for m in DET}
S['friedman_ulb'] = {}
for metric in ['f1', 'pr_auc', 'roc_auc']:
    chi2, p, pairs = ev.friedman_wilcoxon(upf, metric, DET)
    S['friedman_ulb'][metric] = {'chi2': float(chi2), 'p': float(p), 'pairs': pairs.to_dict('records')}
    print(metric, 'Friedman chi2 = %.2f, p = %.3g' % (chi2, p)); print(pairs.round(4))

f1 Friedman chi2 = 22.64, p = 0.000149
  model_a model_b  p_wilcoxon  p_holm
0     KMC     AHC      0.4316  1.0000
1     KMC    GMMC      0.0020  0.0195
2     KMC    FCMC      0.0039  0.0273
3     KMC      IF      0.0137  0.0684
4     AHC    GMMC      0.0098  0.0586
5     AHC    FCMC      0.4316  1.0000
6     AHC      IF      0.4316  1.0000
7    GMMC    FCMC      0.0020  0.0195
8    GMMC      IF      0.0020  0.0195
9    FCMC      IF      0.9219  1.0000
pr_auc Friedman chi2 = 8.72, p = 0.0685
  model_a model_b  p_wilcoxon  p_holm
0     KMC     AHC      0.4316  1.0000
1     KMC    GMMC      0.3223  1.0000
2     KMC    FCMC      0.0020  0.0195
3     KMC      IF      0.3223  1.0000
4     AHC    GMMC      0.6953  1.0000
5     AHC    FCMC      0.4316  1.0000
6     AHC      IF      0.1055  0.9492
7    GMMC    FCMC      0.3223  1.0000
8    GMMC      IF      0.5566  1.0000
9    FCMC      IF      0.3223  1.0000
roc_auc Friedman chi2 = 21.52, p = 0.00025
  model_a model_b  p_wilcoxon  p_holm
0   

### Fraud share of the clusters (K = 2, training folds)

In [11]:
cf = pd.read_csv(R / 'ulb_cluster_fraud_rates_K2.csv')
cf['share'] = cf['mean']
S['cluster_fraud'] = {m: {'max_rate_per_fold': cf[cf.model == m].groupby('fold')['mean'].max().round(4).tolist()} for m in MODELS}
cf.pivot_table(index=['model', 'fold'], columns='cluster', values=['mean', 'size'])

mean                size          
cluster            0         1         0         1
model fold                                        
AHC   1     0.001730  0.000000  256063.0     263.0
      2     0.001730  0.000000  256046.0     280.0
      3     0.006791  0.001717     589.0  255737.0
      4     0.002008  0.001728     498.0  255828.0
      5     0.001730  0.000000  256142.0     184.0
      6     0.001722  0.004808  256118.0     208.0
      7     0.000863  0.850000  256066.0     260.0
      8     0.001124  0.803109  256134.0     193.0
      9     0.001697  0.016129  255769.0     558.0
      10    0.001703  0.021212  255997.0     330.0
FCMC  1     0.001600  0.001860  129414.0  126912.0
      2     0.001800  0.001654  130583.0  125743.0
      3     0.001867  0.001591  127467.0  128859.0
      4     0.001751  0.001704  133102.0  123224.0
      5     0.001722  0.001734  124845.0  131481.0
      6     0.001926  0.001505  133427.0  122899.0
      7     0.001874  0.001576  127541.0  128785.0
      8     0.001702  0.001754  125754.0  130573.0
      9     0.001567  0.001894  129581.0  126746.0
      10    0.001790  0.001668  126227.0  130100.0
GMMC  1     0.000333  0.004265  165362.0   90964.0
      2     0.000302  0.004328  165514.0   90812.0
      3     0.013193  0.000199   30168.0  226158.0
      4     0.004324  0.000308   90661.0  165665.0
      5     0.000332  0.004275  165576.0   90750.0
      6     0.000333  0.004254  165351.0   90975.0
      7     0.004288  0.000320   90720.0  165606.0
      8     0.002452  0.000417  165189.0   91138.0
      9     0.000326  0.004283  165507.0   90820.0
      10    0.000115  0.001982   34807.0  221520.0
KMC   1     0.001926  0.001597  102272.0  154054.0
      2     0.001926  0.001599  101228.0  155098.0
      3     0.000100  0.002028   39820.0  216506.0
      4     0.001947  0.001584  101685.0  154641.0
      5     0.001400  0.003808  221400.0   34926.0
      6     0.001981  0.000086  221577.0   34749.0
      7     0.000125  0.002020   39950.0  216376.0
      8     0.001986  0.000086  221560.0   34767.0
      9     0.001983  0.000115  221430.0   34897.0
      10    0.000115  0.001982   34807.0  221520.0

## Do internal indices predict detection? (ULB, 4 algorithms x K = 2 to 6)

In [12]:
uk = pd.concat([u, pd.read_csv(R / 'ulb_per_fold_K3-4-5-6.csv')]); uk = uk[uk.model != 'IF']
cfg = uk.groupby(['model', 'K'])[['SS', 'CHI', 'DBI', 'f1', 'pr_auc', 'roc_auc', 'recall', 'precision']].mean()
S['ulb_k_configs'] = {f'{m}_K{k}': {c: float(v) for c, v in r.items()} for (m, k), r in cfg.iterrows()}
S['spearman_index_vs_detection'] = {}
for idx in ['SS', 'CHI', 'DBI']:
    for det in ['pr_auc', 'f1', 'roc_auc']:
        r_, p_ = stats.spearmanr(cfg[idx], cfg[det])
        S['spearman_index_vs_detection'][f'{idx}_{det}'] = {'rho': float(r_), 'p': float(p_)}
        print(f'{idx} vs {det}: rho = {r_:.3f}, p = {p_:.3f}')
S['best_SS_config'] = str(cfg['SS'].idxmax()); S['best_prauc_config'] = str(cfg['pr_auc'].idxmax())
S['best_f1_config'] = str(cfg['f1'].idxmax())
cfg.round(4)

SS vs pr_auc: rho = -0.179, p = 0.451
SS vs f1: rho = -0.164, p = 0.490
SS vs roc_auc: rho = -0.050, p = 0.834
CHI vs pr_auc: rho = 0.292, p = 0.211
CHI vs f1: rho = -0.155, p = 0.514
CHI vs roc_auc: rho = 0.223, p = 0.345
DBI vs pr_auc: rho = 0.488, p = 0.029
DBI vs f1: rho = 0.208, p = 0.380
DBI vs roc_auc: rho = 0.018, p = 0.939


SS        CHI     DBI      f1  pr_auc  roc_auc  recall  precision
model K                                                                       
AHC   2  0.7264  4631.9584  1.0953  0.0570  0.1607   0.9533  0.7871     0.0296
      3  0.6511  4441.6021  1.1832  0.0524  0.1544   0.9521  0.7111     0.0272
      4  0.5843  4601.8393  1.3165  0.0505  0.1517   0.9511  0.6458     0.0263
      5  0.5294  4826.5057  1.2861  0.0544  0.1594   0.9519  0.6458     0.0284
      6  0.4951  4893.1023  1.3032  0.0584  0.1673   0.9527  0.6438     0.0306
FCMC  2  0.0452  8069.5248  4.9287  0.0590  0.1739   0.9542  0.8739     0.0306
      3  0.0051  5251.5355  5.2542  0.0594  0.1739   0.9542  0.8800     0.0307
      4 -0.0195  3989.4310  4.9675  0.0590  0.1739   0.9542  0.8759     0.0305
      5 -0.0252  3427.8398  5.0274  0.0594  0.1739   0.9542  0.8780     0.0308
      6 -0.0334  2588.4631  5.1026  0.0586  0.1739   0.9542  0.8759     0.0303
GMMC  2  0.1227  6813.2160  5.2229  0.0396  0.1934   0.9380  0.8394     0.0203
      3  0.0658  7118.9481  4.8930  0.0400  0.2193   0.9334  0.7948     0.0205
      4  0.0476  6804.5287  4.5519  0.0346  0.1461   0.9355  0.7051     0.0178
      5  0.0289  6277.9785  4.4361  0.0370  0.2612   0.9316  0.7129     0.0190
      6  0.0252  5899.0583  4.4226  0.0347  0.1412   0.9271  0.6300     0.0178
KMC   2  0.0751  9524.5366  3.5028  0.0621  0.1745   0.9548  0.8740     0.0322
      3  0.0740  9270.4482  3.5963  0.0575  0.1741   0.9548  0.8760     0.0297
      4  0.0691  9103.5131  3.4843  0.0566  0.1740   0.9553  0.8719     0.0293
      5  0.0738  8814.8053  3.1340  0.0524  0.1613   0.9563  0.8259     0.0271
      6  0.0617  8506.5643  2.9109  0.0380  0.0887   0.9513  0.5372     0.0197

In [13]:
json.dump(S, open(R / 'summary_round4.json', 'w'), indent=1, default=float)
print('saved', R / 'summary_round4.json')

saved ../results/round4/summary_round4.json
